<img src="https://raw.githubusercontent.com/mjcowley/astrolabs/main/Images/qarg_bw_dark.png" alt="QUT Astrophysics" width="350"/>

# AstroLab C11: Surveying the Universe

**PVB220 Cosmology** | Dr Michael Cowley | michael.cowley@qut.edu.au

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjcowley/astrolabs/blob/main/AstroLabC11.ipynb)

## Introduction

This final lab brings the semester together. Every distance, age and luminosity you've calculated so far depends on one model of the Universe: **ΛCDM**, the concordance model. In **Part A** you'll test it yourself with real supernovae, measuring how fast the Universe is expanding and discovering that the expansion is speeding up.

In **Part B** you'll work like a survey astronomer, using two surveys led by QUT researchers and their collaborators in the COSMOS field:

* **ZFOURGE**, a deep imaging survey that measured tens of thousands of galaxies through many filters (you met it in AstroLab C9), and
* **ZFIRE**, a spectroscopic survey that followed up ZFOURGE galaxies with the Keck telescope to measure precise redshifts and emission lines.

You'll explore galaxy **scaling relations**, **cross-match** the two surveys (just like TOPCAT, but in Python), test how good photometric redshifts are, find a **cluster of galaxies being born**, and use emission lines to look **inside** distant galaxies.

This lab is a little longer than the others. Have Canvas open alongside it and answer each 🎯 **Canvas checkpoint** as you go.

▶️ Run the cell below first.

In [ ]:
#@title ▶️ Run me first: setup
import os
import urllib.request

# Download the latest helper file (keep the existing copy if we're offline)
try:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/mjcowley/astrolabs/main/astrolabs.py",
        "astrolabs.py")
except Exception:
    if not os.path.exists("astrolabs.py"):
        raise

import numpy as np                 # numbers and arrays
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # plotting
import astropy.units as u          # physical units
from astropy.constants import c    # the speed of light
from astropy.coordinates import SkyCoord               # positions on the sky
from astropy.cosmology import FlatLambdaCDM            # the concordance model
import importlib
import astrolabs as al             # helper functions for these labs
importlib.reload(al)               # use the version just downloaded

al.setup()

---
# Part A: The Concordance Model

# Part 1: A Modern Hubble Diagram

In the lecture we saw Hubble's 1929 diagram, which gave $H_0 \approx 500$ km/s/Mpc and a Universe only 2 billion years old, younger than the Earth! Let's do better with **Type Ia supernovae**: exploding white dwarfs that all reach nearly the same peak brightness, making them excellent **standard candles**.

The **Pantheon+** compilation contains about 1,600 Type Ia supernovae. For each one, astronomers measured its redshift and its **distance modulus** $\mu = m - M$ (from PVB210: how much fainter it looks than it would at 10 parsecs).

In [ ]:
supernovae = pd.read_csv(al.get_data("C11/pantheon_plus_supernovae.csv"))

print("Supernovae:", len(supernovae))
supernovae.head()

The distance modulus gives the distance: $d = 10^{(\mu + 5)/5}$ parsecs, or $d = 10^{(\mu - 25)/5}$ Mpc. For nearby supernovae, the recession velocity is $v = cz$.

We'll use supernovae with $0.01 < z < 0.05$: far enough that the galaxies' own motions don't matter much, but near enough that $v = cz$ still works well.

In [ ]:
nearby = supernovae[(supernovae["z"] > 0.01) & (supernovae["z"] < 0.05)]

distance = 10 ** ((nearby["distance_modulus"] - 25) / 5)    # in Mpc
velocity = c.to(u.km / u.s).value * nearby["z"]             # in km/s

# Hubble's law v = H0 d is a straight line through the origin: its slope is H0
H0 = np.sum(velocity * distance) / np.sum(distance ** 2)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(distance, velocity, s=8, alpha=0.5, color=al.COLOURS["blue"], label="Pantheon+ supernovae")
d_line = np.linspace(0, distance.max(), 10)
ax.plot(d_line, H0 * d_line, color=al.COLOURS["red"], lw=2, label=f"v = {H0:.1f} × d")
ax.set_xlabel("Distance (Mpc)")
ax.set_ylabel("Recession velocity (km/s)")
ax.legend()
plt.show()

print(f"Hubble constant: H0 = {H0:.1f} km/s/Mpc")
print(f"Hubble time: 1/H0 = {(1 / (H0 * u.km / u.s / u.Mpc)).to(u.Gyr):.1f}")
al.check("C11_hubble_constant", H0)

> 🎯 **Canvas checkpoint Q1:** What value of the Hubble constant do you get from the nearby supernovae?

That's much smaller than Hubble's 500, giving a Hubble time of about 14 billion years: comfortably older than the Earth (4.5 billion years) and the oldest stars.

💡 A careful analysis (accounting for how expansion changes distances even at these redshifts) gives about 73 km/s/Mpc from these supernovae. Hold that thought for Part 3.

# Part 2: The Accelerating Universe

At larger distances, the history of the expansion matters. In a Universe full of **matter**, gravity should slow the expansion down. In 1998, two teams used distant supernovae to measure how much it had slowed, and found the opposite: the expansion is **speeding up**. They won the 2011 Nobel Prize in Physics.

The plot below compares the supernovae with an **empty** Universe (no matter, no dark energy, so the expansion never changes). Supernovae **above** zero are fainter, and so further away, than in an empty Universe.

▶️ Run the cell below. It starts with a Universe full of matter and no dark energy: $\Omega_M = 1$, $\Omega_\Lambda = 0$.

In [ ]:
al.cosmology_explorer(supernovae)

> 🎯 **Canvas checkpoint Q2:** In a Universe with only matter ($\Omega_M = 1$, $\Omega_\Lambda = 0$), how do the distant supernovae compare with the prediction?

👉 Now adjust both sliders to find the best fit (the smallest mismatch).

> 🎯 **Canvas checkpoint Q3:** Roughly what values of $\Omega_M$ and $\Omega_\Lambda$ fit the supernovae best, and what kind of Universe is that?

💡 The supernovae alone pin down the *difference* between matter and dark energy better than their sum, so a range of combinations fit almost equally well. Combining them with the CMB (which says $\Omega_{\rm total} \approx 1$) gives the concordance values $\Omega_M \approx 0.3$ and $\Omega_\Lambda \approx 0.7$.

# Part 3: The Hubble Tension

Different methods give slightly different values of $H_0$, and the difference has become one of the biggest puzzles in cosmology. Here are the values from the lecture. Each implies a different age for the Universe.

In [ ]:
measurements = {
    "Distance ladder (Cepheids + supernovae)": 73.5,
    "Carnegie-Chicago Hubble Program (Freedman 2025)": 70.4,
    "CMB (Planck, assuming ΛCDM)": 67.4,
}

for method, h0 in measurements.items():
    model = FlatLambdaCDM(H0=h0, Om0=0.3)
    print(f"{method:48s} H0 = {h0}   →   age = {model.age(0):.1f}")

> 🎯 **Canvas checkpoint Q4:** Which measurement of $H_0$ implies the **oldest** Universe?

A bigger $H_0$ means faster expansion, so less time was needed to reach today's size. The ages above use the full concordance model rather than just $1/H_0$, because the expansion hasn't been constant: it slowed down while matter dominated, then sped up once dark energy took over.

🔍 New results from **DESI**, the largest 3D map of the Universe (QUT's Dr Rossana Ruggeri is part of the team), hint that dark energy might not be constant after all. Stay tuned!

---
# Part B: Survey Science with ZFOURGE and ZFIRE

# Part 4: Galaxy Scaling Relations

A **scaling relation** is a tight connection between two properties of galaxies. Scaling relations are powerful because any theory of galaxy formation has to reproduce them. Let's explore two using ZFOURGE in the COSMOS field.

In [ ]:
zfourge = pd.read_csv(al.get_data("C11/zfourge_cosmos.csv"))

print("ZFOURGE galaxies in COSMOS:", len(zfourge))
zfourge.head()

The columns are the galaxy's ID and sky position, its photometric redshift `z_phot`, its best redshift `z` (spectroscopic where available), its stellar mass and star formation rate (both as powers of ten), and its size `r_eff_arcsec` (the radius containing half its light).

We'll use the **specific star formation rate** (sSFR = SFR / mass), which says how fast a galaxy is growing compared with its size. Galaxies with sSFR below $10^{-10.5}$ per year are **quiescent**: they'd need more than 30 billion years to double their mass.

In [ ]:
zfourge["log_ssfr"] = zfourge["log_sfr"] - zfourge["log_mass"]
zfourge["quiescent"] = (zfourge["log_ssfr"] < -10.5) | zfourge["log_sfr"].isna()

print(f"Quiescent galaxies: {zfourge['quiescent'].sum()} of {len(zfourge)}")

## The star-forming main sequence

For star-forming galaxies, more massive galaxies form more stars, along a tight **main sequence**. Let's see how it changes over cosmic time.

In [ ]:
star_forming = zfourge[~zfourge["quiescent"] & (zfourge["log_mass"] > 9)]

fig, ax = plt.subplots(figsize=(9, 6))
for (lo, hi), colour in zip([(0.5, 1.0), (1.0, 2.0), (2.0, 3.0)],
                            [al.COLOURS["blue"], al.COLOURS["teal"], al.COLOURS["red"]]):
    sl = star_forming[(star_forming["z"] > lo) & (star_forming["z"] < hi)]
    ax.scatter(sl["log_mass"], sl["log_sfr"], s=3, alpha=0.15, color=colour)
    # the median SFR in bins of mass shows the trend clearly
    bins = np.arange(9, 11.6, 0.25)
    centres = (bins[:-1] + bins[1:]) / 2
    medians = [sl[(sl["log_mass"] > a) & (sl["log_mass"] < b)]["log_sfr"].median() for a, b in zip(bins[:-1], bins[1:])]
    ax.plot(centres, medians, "o-", color=colour, lw=2.5, label=f"{lo} < z < {hi}")

ax.set_xlabel("Stellar mass, log(M / M☉)")
ax.set_ylabel("Star formation rate, log(SFR / M☉ per year)")
ax.set_ylim(-1, 3)
ax.legend()
plt.show()

> 🎯 **Canvas checkpoint Q5:** For star-forming galaxies of the same mass, how does the star formation rate change at earlier times (higher redshift)?

## The mass–size relation

To turn a galaxy's angular size into kiloparsecs, we need its distance, and that comes from the **concordance model** you tested in Part A!

In [ ]:
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

sizes = zfourge[(zfourge["z"] > 0.5) & (zfourge["z"] < 1.5) & (zfourge["log_mass"] > 9.5)].dropna(subset=["r_eff_arcsec"]).copy()
kpc_per_arcsec = cosmo.kpc_proper_per_arcmin(sizes["z"].values).value / 60
sizes["r_eff_kpc"] = sizes["r_eff_arcsec"] * kpc_per_arcsec

fig, ax = plt.subplots(figsize=(9, 6))
for flag, colour, label in [(False, al.COLOURS["blue"], "Star-forming"), (True, al.COLOURS["red"], "Quiescent")]:
    group = sizes[sizes["quiescent"] == flag]
    ax.scatter(group["log_mass"], group["r_eff_kpc"], s=4, alpha=0.3, color=colour, label=label)
ax.set_yscale("log")
ax.set_ylim(0.3, 30)
ax.set_xlabel("Stellar mass, log(M / M☉)")
ax.set_ylabel("Size, half-light radius (kpc)")
ax.set_title("Galaxies at 0.5 < z < 1.5")
ax.legend()
plt.show()

massive = sizes[(sizes["log_mass"] > 10.5) & (sizes["log_mass"] < 11)]
print(f"Median size at 10^10.5 to 10^11 M☉:  star-forming {massive[~massive['quiescent']]['r_eff_kpc'].median():.1f} kpc,"
      f"  quiescent {massive[massive['quiescent']]['r_eff_kpc'].median():.1f} kpc")

> 🎯 **Canvas checkpoint Q6:** At the same stellar mass, how do the sizes of quiescent and star-forming galaxies compare?

Quiescent galaxies are compact: they're thought to have formed most of their stars quickly in dense, gas-rich centres, then grown slowly afterwards, partly through mergers.

# Part 5: Cross-Matching ZFOURGE and ZFIRE

ZFOURGE estimated redshifts from photometry (brightness through many filters). **ZFIRE** used the MOSFIRE spectrograph on the 10-metre Keck I telescope to measure **spectroscopic** redshifts and emission lines for some of the same galaxies. To compare them, we need to know which ZFIRE galaxy is which ZFOURGE galaxy: we need to **cross-match** the catalogues by position, just as you may have done in TOPCAT.

In [ ]:
zfire = pd.read_csv(al.get_data("C11/zfire_cosmos.csv"))

print("ZFIRE galaxies with secure redshifts:", len(zfire))
zfire.head()

In `astropy`, sky positions are stored as `SkyCoord` objects. `match_to_catalog_sky` finds, for every ZFIRE galaxy, the **nearest** ZFOURGE galaxy and how far apart they are.

In [ ]:
zfire_positions = SkyCoord(ra=zfire["ra"].values * u.deg, dec=zfire["dec"].values * u.deg)
zfourge_positions = SkyCoord(ra=zfourge["ra"].values * u.deg, dec=zfourge["dec"].values * u.deg)

index, separation, _ = zfire_positions.match_to_catalog_sky(zfourge_positions)
separation_arcsec = separation.to(u.arcsec).value

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(separation_arcsec, bins=np.logspace(-3, 1, 41), color=al.COLOURS["blue"])
ax.set_xscale("log")
ax.set_xlabel("Separation to the nearest ZFOURGE galaxy (arcseconds)")
ax.set_ylabel("Number of ZFIRE galaxies")
plt.show()
print(f"Median separation: {np.median(separation_arcsec):.2f} arcseconds")

The histogram has two groups. Most ZFIRE galaxies have a ZFOURGE partner less than 0.3″ away: that's the same galaxy seen by both surveys, with a tiny offset from measuring its position in different images. A smaller group sits 2 to 10″ away, about the typical gap between unrelated galaxies in an image this crowded. Those ZFIRE galaxies *are* in the full ZFOURGE catalogue, but their photometry was flagged as unreliable (for example near a bright star or the edge of the image), so they were removed from our cleaned catalogue. Their nearest neighbour in our table is a different galaxy.

## Choosing a match radius, and testing it

We need a **match radius**: pairs closer than this count as the same galaxy. Too small and we miss real matches; too large and we pick up chance neighbours.

A standard diagnostic is to **shift** one catalogue by a large amount (here 30 arcseconds) and match again. Any "matches" now must be chance alignments, so they tell us the **false-match rate** for our radius.

In [ ]:
# 👉 Try different match radii (in arcseconds)
match_radius = 0.5

shifted = SkyCoord(ra=(zfire["ra"].values + 30 / 3600) * u.deg, dec=zfire["dec"].values * u.deg)
_, shifted_sep, _ = shifted.match_to_catalog_sky(zfourge_positions)

real_matches = (separation_arcsec < match_radius).sum()
false_matches = (shifted_sep.to(u.arcsec).value < match_radius).sum()

print(f"Real matches within {match_radius}″:   {real_matches} of {len(zfire)}")
print(f"Chance matches (shifted): {false_matches}  →  false-match rate ≈ {100 * false_matches / len(zfire):.1f}%")

> 🎯 **Canvas checkpoint Q7:** With a 0.5″ match radius, roughly what percentage of matches would be chance alignments?

💡 Try a radius of 3″: you pick up a few of the far-away pairs, which are the wrong galaxies, and the false-match rate shoots up. A good radius keeps the real matches while keeping chance matches rare.

Now combine the matched rows into one table:

In [ ]:
good = separation_arcsec < match_radius
matched = pd.concat([zfire[good].reset_index(drop=True),
                     zfourge.iloc[index[good]].reset_index(drop=True)], axis=1)

print("Matched galaxies:", len(matched))
matched[["zfire_id", "id", "z_spec", "z_phot", "log_mass"]].head()

# Part 6: How Good Are Photometric Redshifts?

Photometric redshifts are cheap (one image per filter gives thousands at once), while spectra are expensive (one Keck night gives a few dozen). So how much can we trust the photometric ones? The standard measure is $\Delta z / (1 + z) = (z_{\rm phot} - z_{\rm spec}) / (1 + z_{\rm spec})$.

In [ ]:
dz = (matched["z_phot"] - matched["z_spec"]) / (1 + matched["z_spec"])

scatter = 1.48 * np.median(np.abs(dz - np.median(dz)))     # a robust measure of scatter
outliers = np.abs(dz) > 0.15                                # "catastrophic" failures

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(matched["z_spec"], matched["z_phot"], s=20, color=al.COLOURS["blue"], label="Matched galaxies")
ax.scatter(matched["z_spec"][outliers], matched["z_phot"][outliers], s=60, facecolor="none",
           edgecolor=al.COLOURS["red"], lw=2, label="Outliers")
ax.plot([1.5, 4], [1.5, 4], "k--", lw=1)
ax.set_xlim(1.8, 3.6)
ax.set_ylim(1.8, 3.6)
ax.set_xlabel("Spectroscopic redshift (ZFIRE)")
ax.set_ylabel("Photometric redshift (ZFOURGE)")
ax.legend()
plt.show()

print(f"Scatter in Δz/(1+z): {scatter:.3f}")
print(f"Catastrophic outliers: {outliers.sum()} ({100 * outliers.mean():.1f}%)")
al.check("C11_outlier_percent", 100 * outliers.mean())

> 🎯 **Canvas checkpoint Q8:** What percentage of the matched galaxies are catastrophic photometric-redshift outliers?

ZFOURGE's medium-band filters make its photometric redshifts unusually precise: a scatter of about 2% in $(1 + z)$. Even so, spectroscopy reveals things photometry can't, as we'll see next.

# Part 7: A Cluster Being Born

In 2012, ZFOURGE photometric redshifts revealed an overdensity of galaxies in COSMOS at $z \approx 2.1$ (Spitler et al. 2012). ZFIRE spectroscopy then confirmed it as a **protocluster**: a galaxy cluster in the process of forming, seen about 11 billion years ago (Nanayakkara et al. 2016).

Let's compare the redshift distributions from spectroscopy and photometry.

In [ ]:
bins = np.arange(1.9, 2.6, 0.01)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(matched["z_phot"], bins=bins, color=al.COLOURS["grey"], alpha=0.6, label="Photometric redshifts (ZFOURGE)")
ax.hist(matched["z_spec"], bins=bins, color=al.COLOURS["red"], alpha=0.8, label="Spectroscopic redshifts (ZFIRE)")
ax.set_xlabel("Redshift")
ax.set_ylabel("Number of galaxies")
ax.legend()
plt.show()

members = matched[matched["cluster_member"]]
print(f"Cluster members: {len(members)}")
print(f"Spectroscopic redshifts: mean {members['z_spec'].mean():.3f}, spread {members['z_spec'].std():.3f}")
print(f"Photometric redshifts:   mean {members['z_phot'].mean():.3f}, spread {members['z_phot'].std():.3f}")

> 🎯 **Canvas checkpoint Q9:** At what redshift is the protocluster, and why does it stand out so much more clearly in the spectroscopic redshifts?

Here's where the members sit on the sky:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(zfourge["ra"], zfourge["dec"], s=1, alpha=0.1, color=al.COLOURS["grey"], label="All ZFOURGE galaxies")
ax.scatter(matched["ra"].iloc[:, 0][~matched["cluster_member"]], matched["dec"].iloc[:, 0][~matched["cluster_member"]],
           s=25, color=al.COLOURS["blue"], label="ZFIRE: not in the cluster")
ax.scatter(members["ra"].iloc[:, 0], members["dec"].iloc[:, 0], s=40, color=al.COLOURS["red"], edgecolor="k",
           label="ZFIRE: cluster members (z ≈ 2.1)")
ax.invert_xaxis()                     # astronomers draw east to the left
ax.xaxis.set_major_locator(plt.MaxNLocator(4))
ax.set_xlabel("Right ascension (degrees)")
ax.set_ylabel("Declination (degrees)")
ax.set_aspect("equal")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
plt.show()

# Light from the cluster set off when the Universe was this old:
print(f"Age of the Universe at z = 2.095: {cosmo.age(2.095):.1f}")

> 🔍 **Peek inside:** after the merge, `matched` has two `ra` columns (one from each survey), so `matched["ra"].iloc[:, 0]` picks the first. When merging catalogues it's common to rename columns first (for example `ra_zfire`, `ra_zfourge`) to avoid this.

# Part 8: Inside the Galaxies

ZFIRE's spectra also measured **emission lines**: Hα and Hβ from hydrogen, [N II] from nitrogen and [O III] from oxygen. Their ratios reveal what's lighting up the gas.

## The BPT diagram

In AstroLab C9 you met AGN. The **BPT diagram** (named after Baldwin, Phillips and Terlevich) separates gas lit by young stars from gas lit by an AGN, which produces harder radiation and stronger [N II] and [O III]. The dividing line moves with redshift, because galaxies at $z \approx 2$ have more extreme conditions than nearby galaxies.

In [ ]:
bpt = matched.dropna(subset=["flux_halpha", "flux_nii", "flux_hbeta", "flux_oiii"]).copy()
bpt = bpt[(bpt[["flux_halpha", "flux_nii", "flux_hbeta", "flux_oiii"]] > 0).all(axis=1)]   # keep detected lines only
bpt["log_nii_ha"] = np.log10(bpt["flux_nii"] / bpt["flux_halpha"])
bpt["log_oiii_hb"] = np.log10(bpt["flux_oiii"] / bpt["flux_hbeta"])

line_x = np.linspace(-2, 0.45, 300)
above = bpt["log_oiii_hb"] > al.bpt_lines(bpt["log_nii_ha"], redshift=2.1)
above |= bpt["log_nii_ha"] > 0.02 + 0.1833 * 2.1                       # right of the line counts too

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(bpt["log_nii_ha"][~above], bpt["log_oiii_hb"][~above], s=50, color=al.COLOURS["blue"], label="Star-forming")
ax.scatter(bpt["log_nii_ha"][above], bpt["log_oiii_hb"][above], s=50, color=al.COLOURS["red"], label="AGN region")
ax.plot(line_x, al.bpt_lines(line_x, redshift=2.1), "k-", lw=2, label="Dividing line at z = 2.1")
ax.plot(line_x, al.bpt_lines(line_x, redshift=0), "k:", lw=1.5, label="Dividing line today")
ax.set_xlim(-2.0, 0.3)
ax.set_ylim(-0.8, 1.4)
ax.set_xlabel("log([N II] / Hα)")
ax.set_ylabel("log([O III] / Hβ)")
ax.legend(loc="lower left", fontsize=9)
plt.show()

print(f"Galaxies with all four lines: {len(bpt)}")
print(f"In the AGN region: {above.sum()}")

> 🎯 **Canvas checkpoint Q10:** Where do most of the ZFIRE galaxies sit on the BPT diagram, and what does that tell us?

Most of these galaxies are powered by star formation, though a few lie in the AGN region. Notice that the dividing line for $z \approx 2$ sits higher than today's: young galaxies have hotter, denser gas, so star formation alone can produce stronger [O III]. A galaxy just above today's line can still be purely star-forming at $z \approx 2$.

## The mass–metallicity relation

One more scaling relation, this time from spectra. The [N II]/Hα ratio is a measure of the gas's **metallicity** (how much of it is heavier than helium), using $12 + \log({\rm O/H}) = 8.90 + 0.57 \log([{\rm N\,II}]/{\rm H}\alpha)$. For comparison, the Sun has $12 + \log({\rm O/H}) \approx 8.7$.

In [ ]:
metal = matched.dropna(subset=["flux_halpha", "flux_nii", "log_mass"]).copy()
metal = metal[~metal["agn"].str.startswith("yes")]                 # AGN light would confuse the measurement
metal = metal[metal["flux_nii"] > 0]                               # keep detected [N II] only
metal["metallicity"] = 8.90 + 0.57 * np.log10(metal["flux_nii"] / metal["flux_halpha"])

bins = np.arange(9.0, 11.25, 0.5)
centres = (bins[:-1] + bins[1:]) / 2
medians = [metal[(metal["log_mass"] > a) & (metal["log_mass"] < b)]["metallicity"].median() for a, b in zip(bins[:-1], bins[1:])]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(metal["log_mass"], metal["metallicity"], s=25, alpha=0.6, color=al.COLOURS["teal"], label="ZFIRE galaxies")
ax.plot(centres, medians, "o-", color="k", lw=2.5, label="Median")
ax.axhline(8.69, color=al.COLOURS["orange"], ls="--", label="The Sun")
ax.set_xlabel("Stellar mass, log(M / M☉) (from ZFOURGE)")
ax.set_ylabel("Gas metallicity, 12 + log(O/H)")
ax.legend()
plt.show()

More massive galaxies are more chemically enriched: they've turned more of their gas into stars, which forged heavier elements and returned them to the gas when they died. That's the PVB210 story of stellar populations, playing out across whole galaxies. Notice that this plot needed **both** surveys: the mass from ZFOURGE and the metallicity from ZFIRE. That's the power of cross-matching.

---
# The End of the AstroLabs

| Lab | Week | What you did |
| --- | :-: | --- |
| C7 | 2 | Python toolkit |
| C8 | 4 | The Big Bang and the CMB |
| C9 | 6 | Galaxies and their black holes |
| C10 | 8 | Lensing, dark matter and gravitational waves |
| **C11** | 10 | The concordance model and survey science (this lab!) |

### Sparkler ideas from this lab

* The accelerating Universe: what the supernovae tell us
* The Hubble tension: a crack in ΛCDM?
* Photometric or spectroscopic? Testing ZFOURGE with ZFIRE
* A cluster being born at z = 2.1
* The BPT diagram in the early Universe

**Well done, you've finished all the AstroLabs!** Make sure you've answered all the 🎯 Canvas checkpoints, and good luck with your Sparkler.

---
# 🚀 Stretch Activity: Every Match, Not Just the Nearest

*Optional, if you have time left.*

`match_to_catalog_sky` only finds the **nearest** neighbour. Sometimes you want **every** galaxy within a radius, for example to find close pairs that might be merging. `search_around_sky` does exactly that.

In [ ]:
# 👉 Try a radius of 2, 5 or 10 arcseconds
radius = 5 * u.arcsec

i_zfourge, i_zfire, sep, _ = zfire_positions.search_around_sky(zfourge_positions, radius)

neighbours = pd.Series(i_zfire).value_counts()
print(f"Pairs found within {radius}: {len(i_zfire)}")
print(f"ZFIRE galaxies with more than one ZFOURGE neighbour: {(neighbours > 1).sum()}")

🚀 **Extra challenge:** for ZFIRE galaxies with more than one neighbour, are the neighbours at the same photometric redshift (a real close pair) or at very different redshifts (a chance alignment along the line of sight)? Use `zfourge.iloc[...]` to look them up.

💡 Careful: `search_around_sky` returns the indices in the order (catalogue you searched *around*, catalogue you called it *on*), so here the first array indexes ZFOURGE and the second indexes ZFIRE.

---
### Data credits

* Pantheon+ Type Ia supernovae: Scolnic et al. 2022, ApJ, 938, 113; Brout et al. 2022, ApJ, 938, 110 (github.com/PantheonPlusSH0ES/DataRelease). Distance moduli calibrated with SH0ES Cepheids (Riess et al. 2022)
* ZFOURGE COSMOS catalogue: Straatman et al. 2016, ApJ, 830, 51, via VizieR. Star formation rates from UV + IR (Tomczak et al. 2016); sizes from HST imaging (van der Wel et al. 2014)
* ZFIRE COSMOS catalogue: Nanayakkara et al. 2016, ApJ, 828, 21, via VizieR
* Protocluster discovery: Spitler et al. 2012, ApJL, 748, L21
* BPT dividing lines: Kewley et al. 2001, ApJ, 556, 121; Kewley et al. 2013, ApJ, 774, 100
* Metallicity calibration: Pettini & Pagel 2004, MNRAS, 348, L59
* H₀ values: Riess et al. (distance ladder, as quoted in the lecture), Freedman et al. 2025, Planck Collaboration 2020